In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [20]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [21]:
import ollama
import fitz
import re


In [4]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [17]:
!ollama ls

NAME                        ID              SIZE      MODIFIED    
mxbai-embed-large:latest    468836162de7    669 MB    3 days ago     
bge-m3:latest               790764642607    1.2 GB    7 days ago     
nomic-embed-text:latest     0a109f422b47    274 MB    7 weeks ago    


In [18]:
!ollama pull qwen3-vl:8b

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠙ pulling manifest 
pulling ed12a4674d72:   0% ▕                  ▏ 4.1 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   0% ▕                  ▏  17 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  31 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  45 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  52 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  66 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  80 MB/6.1 GB                  pulling manifest 
pulling ed12a4674d72:   1% ▕                  ▏  87 MB/6.1 GB                  pulling manifes

In [22]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
import concurrent
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
    base_url="http://localhost:11434"
)
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [24]:
query = "Finding real-world clinical notes is famously difficult due to HIPAA and privacy regulations, but there are several high-quality open-ish repositories and synthetic alternatives that are standard in the NLP research community. As of 2026, here are the primary sources for clinical notes: The Gold Standards (Credentialed Access) These are real, de-identified clinical notes. While open source in spirit, they require a signed Data Use Agreement (DUA) and completion of a short ethics course (usually via CITI). MIMIC-IV (PhysioNet): The most widely used dataset in the world. It contains over 2 million de-identified clinical notes (discharge summaries, radiology reports, etc.) from the Beth Israel Deaconess Medical Center. Note: The MIMIC-IV-Note module was released as a separate extension to the core database."

In [25]:
rec = text_splitter.split_text(query)

ResponseError: this model does not support embeddings (status code: 501)

In [21]:
len(rec)

2

In [23]:
rec[0]

'Finding real-world clinical notes is famously difficult due to HIPAA and privacy regulations, but there are several high-quality open-ish repositories and synthetic alternatives that are standard in the NLP research community. As of 2026, here are the primary sources for clinical notes: The Gold Standards (Credentialed Access) These are real, de-identified clinical notes. While open source in spirit, they require a signed Data Use Agreement (DUA) and completion of a short ethics course (usually via CITI). MIMIC-IV (PhysioNet): The most widely used dataset in the world. It contains over 2 million de-identified clinical notes (discharge summaries, radiology reports, etc.) from the Beth Israel Deaconess Medical Center.'

In [5]:
from tqdm.auto import tqdm

In [ ]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import concurrent.futures


In [59]:

def load_single_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    return text_splitter.split_documents([doc])

def clean_text(text):
    # Collapse spaced-out characters: "W e  w i l l" -> "We will"
    text = re.sub(r'\b(\w) (\w) ', r'\1\2', text)  # iterative collapse
    # Run it multiple times since each pass only collapses pairs
    for _ in range(20):
        text = re.sub(r'(?<!\w)(\w) (\w)(?!\w)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="mxbai-embed-large",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    pre_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,   # characters, not tokens
        chunk_overlap=300
    )

    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
    
    docs = pre_splitter.split_documents(docs)
        
    semantic_chunks = []
    
    MAX_WORKERS = 20
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            try:
                semantic_chunks.extend(future.result())
            except Exception as e:
                print(f"Skipping a chunk due to error: {e}")  # Don't crash on one bad doc

    final_chunks = []

    for chunk in semantic_chunks:
        if len(chunk.page_content) > 1800:
            final_chunks.extend(pre_splitter.split_documents([chunk]))
        else:
            final_chunks.append(chunk)
    
    for chun in final_chunks:
        chun.page_content = clean_text(chun.page_content)
    return final_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

Starting parallel PDF loading...


Loading PDFs:   0%|          | 0/18 [00:00<?, ?it/s]

Starting parallel chunking with 20 workers...


Splitting documents:   0%|          | 0/3603 [00:00<?, ?it/s]

In [61]:
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import ResponseError

In [62]:
client = chromadb.PersistentClient(path="chroma_db")

ef = OllamaEmbeddingFunction(
    model_name = "mxbai-embed-large",
    url="http://localhost:11434/api/embeddings"
)

In [63]:
collection = client.get_or_create_collection(name="semantic_texts", embedding_function=ef)

In [65]:

embeddings_model = OllamaEmbeddings(
    model="mxbai-embed-large", 
    base_url="http://localhost:11434"
    )

# Re-split any oversized chunks (more aggressive threshold)
safety_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=300)
    
final_chunks = []
for chunk in semantic_chunks_parallel:
    if len(chunk.page_content) > 1700:
        final_chunks.extend(safety_splitter.split_documents([chunk]))
    else:
        final_chunks.append(chunk)

print(f"Before: {len(semantic_chunks_parallel)}, After: {len(final_chunks)}")

texts = [doc.page_content for doc in final_chunks]
metadatas = [doc.metadata for doc in final_chunks]

# Embed one-by-one to avoid any batch issues
all_embeddings = []
for i in tqdm(range(len(texts)), desc="Embedding"):
    try:
        emb = embeddings_model.embed_documents([texts[i]])
        all_embeddings.extend(emb)
    except ResponseError:
        print(f"Chunk {i} too long ({len(texts[i])} chars), sub-splitting...")
        sub = safety_splitter.split_text(texts[i])
        # Just embed the first sub-chunk to keep 1:1 mapping
        all_embeddings.extend(embeddings_model.embed_documents([sub[0]]))

assert len(all_embeddings) == len(texts), "Mismatch!"
batch_size = 5000
for start in range(0, len(texts), batch_size):
    end = min(start + batch_size, len(texts))
    collection.add(
        ids=[f"chunk_{i}" for i in range(start, end)],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
        embeddings=all_embeddings[start:end]
    )
    print(f"Added batch {start}-{end}")

Before: 7095, After: 7095


Embedding:   0%|          | 0/7095 [00:00<?, ?it/s]

Added batch 0-5000
Added batch 5000-7095


In [66]:
query = "What is the misfit function?"

#print(len(query))
embeddings_model = OllamaEmbeddings(
        model="mxbai-embed-large", 
        base_url="http://localhost:11434"
        )
safety_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1700, 
        chunk_overlap=200
        )

query_to_chunks = []
if len(query) >1800:
    query_chunks = safety_splitter.split_documents(query)
    query_to_chunks.extend(query_chunks)
else:
    query_to_chunks.extend(query)


query_emb = embeddings_model.embed_documents(query_to_chunks)
    
dense_results = collection.query(query_embeddings= query_emb, n_results=5)
    
dense_ids = [int(id.split("_")[1]) for id in dense_results["ids"][0]]

for i in dense_ids:
    print(texts[i])
    #dic = metadatas[i]
    #print(dic)

For Tel , 15 K,
While such an increase may lead to the appear-
cies on the band gap. The vacancies are simulated by impos- ing a high onsite potential in the target orbitals—here the s and p orbitals of various S atoms in the 3 × 4 reproduction cell. The very occurrence of the unique peaks produced by the pres- ence of disorder illustrates how the CAs with different disorder concentrations provide unique curves that populate the space of possible input signals.
We
We


In [67]:
from rank_bm25 import BM25Okapi
import pickle

In [68]:
tokenized_corpus = [doc.lower().split(" ") for doc in texts]
bm25 = BM25Okapi(tokenized_corpus)
with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)

In [69]:
"""example"""

query = "universal conductance"
tokenized_query = query.lower().split(" ")
scores = bm25.get_scores(tokenized_query)
top_10 = np.argsort(scores)[::-1][:10]


In [70]:
len(query)

21

In [71]:
for i , inx in enumerate(top_10):
    if scores[inx] >12:
        print(scores[inx])
        dic = metadatas[inx]
        print(dic['source'])

12.172096698490826
raw/p1.pdf
12.102764637338794
raw/p4.pdf


In [72]:
def hybrid_search(query, top_k, dense_weight=0.6, sparse_weight=0.4):
    k_cand = max(15, top_k * 3)

    # --- Sparse (BM25) retrieval ---
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    sparse_top = np.argsort(bm25_scores)[::-1][:k_cand]

    # --- Dense (embedding) retrieval ---
    embeddings_model = OllamaEmbeddings(
        model="mxbai-embed-large", 
        base_url="http://localhost:11434"
    )

    # embed_query expects a single string, not a list
    query_emb = embeddings_model.embed_query(query)

    dense_results = collection.query(
        query_embeddings=[query_emb], 
        n_results=k_cand,
        include=["documents", "metadatas", "distances"]
    )

    dense_ids = [int(id.split("_")[1]) for id in dense_results["ids"][0]]
    # ChromaDB distances: lower = more similar (L2) → convert to similarity
    dense_dist = dense_results["distances"][0]
    max_dense = max(dense_dist) if dense_dist else 1.0
    dense_sim = {idx: 1 - (d / max_dense) for idx, d in zip(dense_ids, dense_dist)}

    # --- Normalise BM25 scores for the sparse candidates ---
    max_bm25 = bm25_scores[sparse_top[0]] if bm25_scores[sparse_top[0]] > 0 else 1.0
    sparse_sim = {int(idx): bm25_scores[idx] / max_bm25 for idx in sparse_top}

    # --- Weighted fusion ---
    all_candidates = set(dense_sim.keys()) | set(sparse_sim.keys())
    fused = {}
    for idx in all_candidates:
        d_score = dense_sim.get(idx, 0.0) * dense_weight
        s_score = sparse_sim.get(idx, 0.0) * sparse_weight
        fused[idx] = d_score + s_score

    # --- Rank and return top_k ---
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, score in ranked:
        results.append({
            "chunk_index": idx,
            "text": texts[idx],
            "metadata": metadatas[idx],
            "hybrid_score": round(score, 4),
            "dense_score": round(dense_sim.get(idx, 0.0), 4),
            "sparse_score": round(sparse_sim.get(idx, 0.0), 4),
        })

    return results


In [73]:
results = hybrid_search("universal conductance ﬂuctuations", top_k=5, dense_weight=0.5, sparse_weight=0.5)
for r in results:
    print(f"[{r['hybrid_score']}] {r['text']}...")


[0.5] A. Lee and A. Douglas Stone. Universal conductance ﬂuctuations in metals.Phys. Rev. Lett., 55:1622–1625, Oct 1985. [51] P. A. Lee, A. Douglas Stone, and H. Fukuyama. Universal conductance ﬂuctuations in metals:...
[0.4684] Universal conductance ﬂuctuations (UCF) help one to obtain accurate results for⟨Γ(E,n )⟩at a...
[0.4605] [62] P. A. Lee, A. D. Stone, Universal conductance ﬂuctuations in metals, Phys. Rev....
[0.4605] Universal conductance ﬂuctuations (UCF) can be used to estimate values of M required for...
[0.401] nanomaterials. Inverse Problems, 29:015006, 2013. [50] P. A. Lee and A. Douglas Stone. Universal conductance ﬂuctuations in metals.Phys. Rev. Lett., 55:1622–1625, Oct 1985. [51] P. A. Lee, A. Douglas Stone, and H. Fukuyama. Universal conductance ﬂuctuations in metals: Eﬀects of ﬁnite temperature, interactions, and magnetic ﬁeld.Phys. Rev. B, 35:1039–1070, Jan 1987. [52] Harold U. Baranger and Pier A....


In [76]:
import ollama

def rag_answer(query, top_k=10, dense_weight=0.6, sparse_weight=0.4):
    # 1. Retrieve context via hybrid search
    results = hybrid_search(query, top_k=top_k, 
                            dense_weight=dense_weight, 
                            sparse_weight=sparse_weight)

    # 2. Build context block from retrieved chunks
    context_block = ""
    for i, r in enumerate(results):
        source = r["metadata"].get("source", "unknown")
        context_block += f"--- Chunk {i+1} [source: {source}] ---\n"
        context_block += r["text"] + "\n\n"
    print(context_block)
    # 3. Prompt the LLM
    system_prompt = (
        "You are an expert scientific AI assistant. "
        "Answer the user's question using ONLY the provided context. "
        "Cite which chunk(s) you drew each part of the answer from. "
        "Generate a figure to describe the result where applicable"
        "look for mathematical descriptions"
        "If the context does not contain enough information, say so clearly."
    )

    user_prompt = (
        f"## Context\n{context_block}\n"
        f"## Question\n{query}\n\n"
        "Provide a detailed, well-structured answer."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    # 4. Stream the response
    print(f"Query: {query}\n{'='*60}")
    full_response = ""
    for chunk in ollama.chat(model="qwen3-vl:8b", messages=messages, stream=True):
        token = chunk["message"]["content"]
        full_response += token
        print(token, end="", flush=True)
    
    print(f"\n{'='*60}")
    return full_response, results


In [77]:
answer, sources = rag_answer("What is the misfit function?")


--- Chunk 1 [source: raw/p1.pdf] ---
DFT-based misﬁt function Although the approach described above is undoubt- edly a good strategy to illustrate what can be achieved by carrying out the CA calculation within the TB model,

--- Chunk 2 [source: raw/p4.pdf] ---
Decoding the conductance of disordered nanostructures: a quantum inverse problem 2 can one infer about the Hamiltonian components from that information alone? To make matters worse, what if the system is made of a heavily disordered material?

--- Chunk 3 [source: raw/p1.pdf] ---
what the real impurity concentration that generated the Hamiltonian is but also the exact locations of the ran- domly placed impurities. Interestingly, as the concen- tration increases we ﬁnd a growing number of impurities that are adjacent to one another forming pairs that may actasiftheywereisolatedscatterers.

--- Chunk 4 [source: raw/p1.pdf] ---
see thatαis very small for all concentrations below 8%, clearly indicating that our inversion tool is ver